
# 从 FX 到 Dynamo，再到 `torch.compile` —— 实战 Notebook

本 Notebook 覆盖：
1) **FX 基础**：`symbolic_trace`、查看/修改图、简单图变换  
2) **TorchDynamo**：自动捕获 Python 代码并交给自定义 backend  
3) **`torch.compile`**（Inductor）: 一行代码编译加速与基准测试  


In [ ]:

import torch, time, math, os, sys
import torch.nn as nn
import torch.nn.functional as F
from torch.fx import symbolic_trace, GraphModule
from torch.fx.passes.shape_prop import ShapeProp
from torch.fx.subgraph_rewriter import replace_pattern

print("PyTorch version:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
torch.manual_seed(0)



## A. FX 基础：trace、查看、修改图
我们先搭一个小模型，演示如何：  
- 用 `symbolic_trace` 把 `nn.Module` 变成 FX Graph  
- 打印图并跑一遍推理  
- **修改图**（把 ReLU 换成 SiLU）并重新编译执行  


In [ ]:

class TinyMLP(nn.Module):
    def __init__(self, in_dim=8, hidden=16, out_dim=4):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

model = TinyMLP().to(device).eval()
x = torch.randn(2, 8, device=device)

traced: GraphModule = symbolic_trace(model)
print("=== 原始 FX Graph ===")
print(traced.graph)
print("\n运行输出：", traced(x).shape)


In [ ]:

# 把 call_module: act(ReLU) 替换成 SiLU
# 方法：直接改子模块并修补图中的 target
import copy
gm = copy.deepcopy(traced)

# 1) 替换子模块
gm.act = nn.SiLU()

# 2) 由于 target 名称未变（仍是 'act'），图里无需改 target；
#    但是一种更通用的写法是直接编辑 graph.nodes（这里给出参考做法）

print("=== 修改后（ReLU -> SiLU）的运行输出 ===")
gm.recompile()
out = gm(x)
print(out.shape)



### A.1 基于模板匹配的图变换（`subgraph_rewriter`）
我们用一个简单模式：`Linear -> ReLU`，把它替换成 `Linear -> GELU`。  
这不是真正的 kernel 融合，只是演示 **如何在图上做模式替换**。  


In [ ]:

# 定义匹配模式与替换模式
class Pattern(nn.Module):
    def __init__(self, in_dim=8, hidden=16):
        super().__init__()
        self.fc = nn.Linear(in_dim, hidden)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.act(self.fc(x))

class Replacement(nn.Module):
    def __init__(self, in_dim=8, hidden=16):
        super().__init__()
        self.fc = nn.Linear(in_dim, hidden)
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(self.fc(x))

pattern = symbolic_trace(Pattern())
replacement = symbolic_trace(Replacement())

gm2 = symbolic_trace(TinyMLP())

print("=== 图变换前 ===")
gm2.graph.print_tabular()

n_before = len(list(gm2.graph.nodes))

replace_pattern(gm2, pattern, replacement)

gm2.recompile()
print("\n=== 图变换后 ===")
gm2.graph.print_tabular()
n_after = len(list(gm2.graph.nodes))
print(f"节点数: {n_before} -> {n_after}")



## B. TorchDynamo：自动捕获 Python 代码 → 交给自定义 Backend
`TorchDynamo` 会拦截 Python 帧，识别张量运算并生成 FX Graph。  
我们实现一个 **自定义 backend**：打印捕获到的 FX 图，然后原样返回可执行模块。  


In [ ]:

import torch._dynamo as dynamo

def print_graph_backend(gm: GraphModule, example_inputs):
    print("=== Dynamo 捕获的 FX Graph（print_tabular） ===")
    gm.graph.print_tabular()
    # 也可以在这里对 gm 做变换或交给别的编译器
    return gm  # 返回可执行的 GraphModule

@dynamo.optimize(print_graph_backend)
def dyn_func(a, b):
    # 混合一点 Python 控制流，看看是否能被捕获
    if a.shape[0] % 2 == 0:
        x = torch.add(a, b)
    else:
        x = torch.sub(a, b)
    return F.relu(x @ x.transpose(-1, -2))

A = torch.randn(4, 8, device=device)
B = torch.randn(4, 8, device=device)
out_dyn = dyn_func(A, B)
print("dyn_func 输出形状：", out_dyn.shape)


In [ ]:

# 也可以用 export 拿到静态图（捕获的 GraphModule 和示例输入）
try:
    gm_export, guards = torch._dynamo.export(dyn_func, A, B)
    print("=== export 得到的 Graph ===")
    gm_export.graph.print_tabular()
except Exception as e:
    print("export 可能在当前版本不可用或接口变动：", e)



## C. `torch.compile`（Inductor）一键加速与基准测试
PyTorch 2+ 提供 `torch.compile(model_or_fn, backend="inductor")`：  
- 首次运行会进行捕获、编译与缓存  
- 之后相同形状/分支下直接复用已编译代码  

我们做一个轻量基准：


In [ ]:

def bench(fn, iters=100, warmup=20):
    # 预热
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t1 = time.perf_counter()
    return (t1 - t0) / iters

class ToyNet(nn.Module):
    def __init__(self, d=1024):
        super().__init__()
        self.l1 = nn.Linear(d, d, bias=False)
        self.l2 = nn.Linear(d, d, bias=False)
        self.l3 = nn.Linear(d, d, bias=False)
    def forward(self, x):
        x = self.l1(x)
        x = F.relu(x)
        x = self.l2(x)
        x = torch.sin(x) + torch.cos(x)
        x = self.l3(x)
        return x

net = ToyNet().to(device).eval()
inp = torch.randn(256, 1024, device=device)

# eager
def run_eager():
    with torch.inference_mode():
        net(inp)

t_eager = bench(run_eager, iters=50, warmup=10)
print(f"Eager 平均每次: {t_eager*1e3:.3f} ms")

# compile（如果不可用则降级提示）
compiled_net = None
if hasattr(torch, "compile"):
    try:
        compiled_net = torch.compile(net)  # 默认 backend=inductor
        def run_compiled():
            with torch.inference_mode():
                compiled_net(inp)
        t_comp = bench(run_compiled, iters=50, warmup=10)
        print(f"compile 平均每次: {t_comp*1e3:.3f} ms, 加速比≈ {t_eager/max(t_comp,1e-9):.2f}x")
    except Exception as e:
        print("torch.compile 在当前环境不可用或触发限制：", e)
else:
    print("当前 PyTorch 版本不支持 torch.compile")



### 备注
- 真实性能取决于环境（CPU/GPU、BLAS、驱动、PyTorch 版本）。
- `torch.compile` 首次运行包含编译开销；上面的基准已做预热以更贴近 steady-state。
- 你可以把 `ToyNet` 和输入规模改大/改小，观察不同形状对编译与融合的影响。
- 若要进一步自定义编译流程，可把 Dynamo 的 backend 换成你自己的函数，在里面对 FX 图执行优化或把它交给其它编译器（如 TVM / TensorRT 等）。
